<h1><center>General Info</center></h1>

This part covers basic feature selection pipeline implementation.

**1)** Firstly, the data are split in startified manner `train_test_split` into train, validate and test

**2)** Feature encoding with `ColumnTransformer` and `OneHotEncoder` 

**3)** `VarianceThreshold` was used as a first approach to filter our non-informative features

**4)** Then `BorutaPy` via `BalancedRandomForestClassifier` selects the final feature set

**5)** Export to `.parquet` is used for the final feauture set

---

<h1><center>Imports</center></h1>

In [1]:
import os, sys, pickle

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split  
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder      
from sklearn.feature_selection import VarianceThreshold 

from imblearn.ensemble import BalancedRandomForestClassifier

from boruta import BorutaPy

<h1><center>CWD and Path</center></h1>

In [2]:
print("Current working dir:", os.getcwd())

Current working dir: /Users/ddi/git_hub/pros_project/src/notebooks


In [3]:
print("Initial Python path:", sys.path)

sys.path.append(os.getcwd().split('notebooks')[0])

print("\nPython path after append:", sys.path)

Initial Python path: ['/Library/Frameworks/Python.framework/Versions/3.10/lib/python310.zip', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/lib-dynload', '', '/Users/ddi/python_venv/pros/lib/python3.10/site-packages']

Python path after append: ['/Library/Frameworks/Python.framework/Versions/3.10/lib/python310.zip', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/lib-dynload', '', '/Users/ddi/python_venv/pros/lib/python3.10/site-packages', '/Users/ddi/git_hub/pros_project/src/']


<h1><center>GLOBALS</center></h1>

In [4]:
MAJOR_VERSION = 1
MINOR_VERSION = 0
PATCH_VERSION = 0

IMPORT_PATH = '../data/imports/customer_churn_dataset_250k.parquet'
EXPORT_PATH = f'../data/exports/features/selected_features_v{MAJOR_VERSION}.{MINOR_VERSION}.{PATCH_VERSION}.pkl'

FLAG_EXPORT = True

<h1><center>Init</center></h1>

In [5]:
from utils.base_classes import FeatureSelection

feature_selection = FeatureSelection()

<h1><center>Data Load</center></h1>

In [6]:
df = feature_selection.load_data(IMPORT_PATH)
df

,customer_id,age,gender,region,contract_type,tenure_months,internet_service,monthly_charges,data_usage_gb,avg_call_duration_minutes,support_calls,late_payments,payment_method,satisfaction_score,churn
0,1,56,Female,Asia,Month-to-Month,42,5G,49.49,244.86,4.92,2,1,Digital Wallet,5,0
1,2,69,Male,Europe,Two Year,23,DSL,92.74,211.98,4.04,4,0,Debit Card,3,0
2,3,46,Female,South America,Month-to-Month,64,5G,26.95,179.35,6.92,3,0,Bank Transfer,7,0
3,4,32,Female,North America,Month-to-Month,38,Fiber Optic,69.06,5.00,4.19,5,0,Credit Card,9,1
4,5,60,Male,Asia,One Year,57,Fiber Optic,64.55,193.71,4.09,2,1,Debit Card,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,249996,18,Female,North America,One Year,10,DSL,53.61,131.46,5.84,1,2,Credit Card,1,0
249996,249997,34,Female,Africa,Month-to-Month,70,None,28.55,85.89,6.58,1,0,Debit Card,5,0
249997,249998,32,Male,North America,Month-to-Month,28,Fiber Optic,66.12,99.72,4.50,3,2,Debit Card,1,1
249998,249999,34,Female,Africa,One Year,26,Fiber Optic,95.16,142.50,6.25,3,0,Debit Card,10,0


In [7]:
df.dtypes

customer_id                    int64
age                            int64
gender                        object
region                        object
contract_type                 object
tenure_months                  int64
internet_service              object
monthly_charges              float64
data_usage_gb                float64
avg_call_duration_minutes    float64
support_calls                  int64
late_payments                  int64
payment_method                object
satisfaction_score             int64
churn                          int64
dtype: object

In [8]:
df['gender'].unique()

array(['Female', 'Male'], dtype=object)

In [9]:
df['region'].unique()

array(['Asia', 'Europe', 'South America', 'North America', 'Africa'],
      dtype=object)

In [10]:
df['contract_type'].unique()

array(['Month-to-Month', 'Two Year', 'One Year'], dtype=object)

In [11]:
df['internet_service'].unique()

array(['5G', 'DSL', 'Fiber Optic', None], dtype=object)

In [12]:
df['payment_method'].unique()

array(['Digital Wallet', 'Debit Card', 'Bank Transfer', 'Credit Card'],
      dtype=object)

In [13]:
df['internet_service'] = df['internet_service'].fillna('None')

<h1><center>Stratify Split</center></h1>

In [14]:
X_train, X_val, X_test, y_train, y_val, y_test = feature_selection.stratify_split(df=df,
                                                                                  target='churn', 
                                                                                  test_size=0.1,
                                                                                  id_col='customer_id')


NaN check:

 customer_id                  0
age                          0
gender                       0
region                       0
contract_type                0
tenure_months                0
internet_service             0
monthly_charges              0
data_usage_gb                0
avg_call_duration_minutes    0
support_calls                0
late_payments                0
payment_method               0
satisfaction_score           0
churn                        0
dtype: int64

Unique labels available: [0 1]

Stratify criterion satisfied. Class ratio for label 0: 0.69

Stratify criterion satisfied. Class ratio for label 1: 0.31


<h1><center>One Hot Encoding</center></h1>

In [15]:
X_train, X_val, X_test = feature_selection.feature_encoding(X_train=X_train,
                                                            X_val=X_val,
                                                            X_test=X_test,
                                                            categories=['gender', 'region', 'contract_type',
                                                                        'internet_service', 'payment_method'])

<h1><center>Variance Threshold</center></h1>

In [16]:
X_train, X_val, X_test = feature_selection.variance_threshold(X_train=X_train,
                                                              X_val=X_val,
                                                              X_test=X_test,
                                                              threshold=1e-1)


Filtered features with VarianceThreshold and threshold 0.1:

 {'internet_service_None', 'region_Africa', 'region_South America'}


<h1><center>Boruta Filtration</center></h1>

In [17]:
X_train, X_val, X_test = feature_selection.run_boruta(X_train=X_train,
                                                      X_val=X_val,
                                                      X_test=X_test,
                                                      y_train=y_train,
                                                      n_estimators=99,
                                                      criterion='gini',
                                                      max_iter=20,
                                                      perc=0.9)

Iteration: 	1 / 20
Confirmed: 	0
Tentative: 	23
Rejected: 	0
Iteration: 	2 / 20
Confirmed: 	0
Tentative: 	23
Rejected: 	0
Iteration: 	3 / 20
Confirmed: 	0
Tentative: 	23
Rejected: 	0
Iteration: 	4 / 20
Confirmed: 	0
Tentative: 	23
Rejected: 	0
Iteration: 	5 / 20
Confirmed: 	0
Tentative: 	23
Rejected: 	0
Iteration: 	6 / 20
Confirmed: 	0
Tentative: 	23
Rejected: 	0
Iteration: 	7 / 20
Confirmed: 	0
Tentative: 	23
Rejected: 	0
Iteration: 	8 / 20
Confirmed: 	20
Tentative: 	3
Rejected: 	0
Iteration: 	9 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	10 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	11 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	12 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	13 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	14 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	15 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	16 / 20
Confirmed: 	20
Tentative: 	2
Rejected: 	1
Iteration: 	17 / 

<h1><center>Export to .pkl</center></h1>

In [18]:
if FLAG_EXPORT:
    feature_selection.pickle_export(path=EXPORT_PATH,
                                    file=feature_selection.selected_features_boruta)